# Meridian: intent model fine-tune on Kaggle (GPU)
Full recipe: embeddings unfrozen, 4 epochs. Needs the Meridian source uploaded as a Kaggle dataset named `meridian-src` (the whole project folder, zipped), and **Internet on** (for Banking77 and DistilBERT).
Settings: Accelerator = GPU T4 x2 or P100. Output: `/kaggle/working/ml/intent/artifacts/distilbert_intent.pt` and metrics printed at the end.

In [ ]:
import os
import shutil
import sys

SRC = "/kaggle/input/meridian-src"
WORK = "/kaggle/working/meridian"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(SRC, WORK, ignore=shutil.ignore_patterns(".venv*", "mlruns", "mlartifacts", "*.db"))
os.chdir(WORK)
sys.path.insert(0, WORK)
print(os.getcwd(), os.listdir(".")[:12])

In [ ]:
!pip install -q transformers datasets scikit-learn pandas pyarrow mlflow rank-bm25 pandera scipy
import torch

print(
    "cuda",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

In [ ]:
from ml.intent.finetune import TrainConfig, run

cfg = TrainConfig(freeze_embeddings=False, epochs=4)
metrics = run(cfg)
print(metrics)

In [ ]:
import json
import os
import shutil

os.makedirs('/kaggle/working/results', exist_ok=True)
with open('/kaggle/working/results/finetune_kaggle.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
shutil.copy('ml/intent/artifacts/distilbert_intent.pt', '/kaggle/working/distilbert_intent.pt')
print('saved: finetune_kaggle.json and distilbert_intent.pt')